# Fundamentals 00.2 - Runtime Bedrock Provider API

Objetivo: probar la ruta `bedrock-runtime` de forma aislada antes de construir tools, agents, systems o graphs.

Este notebook responde tres preguntas:

1. Que `RuntimeConfig` se declara con `provider="bedrock-runtime"`.
2. Que `SchedulerConfig` protege la ejecucion.
3. Que AWS se valida por capas: `sts`, `bedrock` y `bedrock-runtime`.

Regla de diseno:

```text
Agentic Systems define como se ve una ejecucion.
Providers definen donde corre.
Bedrock Runtime es el backend AWS de inferencia.
```


## 0) Imports m?nimos

El notebook asume que `agentic-systems` esta instalado en el ambiente activo.


In [ ]:
import json
import os

import agentic_systems as lab

print("agentic_systems:", lab.__name__)


## Escenario did?ctico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso.


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario did?ctico")


## 1) Utilidad segura de impresion


In [ ]:
def show_json(obj, title: str | None = None) -> None:
    if title:
        print(f"\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


## 2) RuntimeConfig y SchedulerConfig para Bedrock

Esta celda no llama a AWS. Solo declara el contrato local que Agentic Systems usara para seleccionar `bedrock-runtime`.


In [ ]:
scheduler = lab.scheduler(
    timeout_s=60,
    max_retries=1,
    max_tool_calls=5,
    max_turns=6,
    max_concurrency=1,
    backoff_s=0.2,
)

REGION = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "us-east-1"
LANGUAGE_MODEL_ID = os.getenv("BEDROCK_LANGUAGE_MODEL_ID", "qwen.qwen3-32b-v1:0")
EMBEDDING_MODEL_ID = os.getenv("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")

runtime = lab.runtime(
    provider="bedrock-runtime",
    model=LANGUAGE_MODEL_ID,
    region=REGION,
    scheduler=scheduler,
    metadata={"purpose": "fundamentals_bedrock_provider_notebook"},
)

lab.show(runtime.describe(), title="Bedrock runtime describe")
lab.show(lab.boto3_session_snapshot(REGION), title="Boto3 session credential provider")

## 3) Configuracion AWS segura

Las llamadas reales estan apagadas por default. En ADA/AWS sandbox, cambia `RUN_BEDROCK_SMOKE_TESTS=True` despues de verificar credenciales.


In [ ]:
RUN_BEDROCK_SMOKE_TESTS = False

show_json(
    {
        "region": REGION,
        "language_model_id": LANGUAGE_MODEL_ID,
        "embedding_model_id": EMBEDDING_MODEL_ID,
        "run_bedrock_smoke_tests": RUN_BEDROCK_SMOKE_TESTS,
        "has_aws_region": bool(os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")),
        "has_aws_profile": bool(os.getenv("AWS_PROFILE")),
    },
    "bedrock config",
)


## 4) Crear clientes boto3: `sts`, `bedrock`, `bedrock-runtime`

`sts` diagnostica identidad, `bedrock` diagnostica metadata/modelos y `bedrock-runtime` ejecuta inferencia.


In [ ]:
if RUN_BEDROCK_SMOKE_TESTS:
    import boto3

    session = boto3.Session(region_name=REGION)
    sts = session.client("sts")
    bedrock = session.client("bedrock")
    bedrock_runtime = session.client("bedrock-runtime")
    print("Clientes AWS creados.")
else:
    sts = bedrock = bedrock_runtime = None
    print("Saltado: clientes boto3 requieren RUN_BEDROCK_SMOKE_TESTS=True.")


## 5) STS: identidad segura


In [ ]:
if RUN_BEDROCK_SMOKE_TESTS:
    identity = sts.get_caller_identity()
    show_json({"account": identity.get("Account"), "arn": identity.get("Arn")}, "sts identity")
else:
    print("Saltado: STS requiere RUN_BEDROCK_SMOKE_TESTS=True.")


## 6) Bedrock control plane: metadata de modelos


In [ ]:
if RUN_BEDROCK_SMOKE_TESTS:
    models = bedrock.list_foundation_models()
    summaries = models.get("modelSummaries", [])
    show_json({"model_count": len(summaries), "first_models": summaries[:3]}, "bedrock models")
else:
    print("Saltado: Bedrock control plane requiere RUN_BEDROCK_SMOKE_TESTS=True.")


## 7) Bedrock Runtime: Converse para lenguaje

Esta es la ruta que respalda `provider="bedrock-runtime"` para modelos de lenguaje.


In [ ]:
if RUN_BEDROCK_SMOKE_TESTS:
    response = bedrock_runtime.converse(
        modelId=LANGUAGE_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "Responde en una frase: que valida este notebook?"}]}],
        inferenceConfig={"maxTokens": 160, "temperature": 0.0},
    )

    text = "".join(
        block.get("text", "")
        for block in response.get("output", {}).get("message", {}).get("content", [])
    )
    print(text)
    show_json({"usage": response.get("usage"), "stopReason": response.get("stopReason")}, "converse trace")
else:
    print("Saltado: Converse requiere RUN_BEDROCK_SMOKE_TESTS=True.")


## 8) Bedrock Runtime: embeddings via `invoke_model`

Este smoke vive a nivel boto3 porque el runtime principal es de lenguaje. La prueba muestra el backend de embeddings sin mezclarlo con agents.


In [ ]:
if RUN_BEDROCK_SMOKE_TESTS:
    body = {
        "inputText": "Agentic Systems valida providers, schedulers y runtime contracts.",
        "dimensions": 1024,
        "normalize": True,
    }

    embedding_response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps(body).encode("utf-8"),
        contentType="application/json",
        accept="application/json",
    )

    payload = json.loads(embedding_response["body"].read())
    embedding = payload.get("embedding", [])
    show_json({"model_id": EMBEDDING_MODEL_ID, "embedding_length": len(embedding), "first_values": embedding[:5]}, "embedding smoke")
else:
    print("Saltado: embeddings requieren RUN_BEDROCK_SMOKE_TESTS=True.")


## 9) Lectura correcta del dise?o

- `bedrock-runtime` es provider/backend de inferencia.
- `sts` y `bedrock` son diagn?stico AWS, no providers de Agentic Systems.
- `runtime.describe()` audita la configuraci?n declarada; `lab.boto3_session_snapshot(...)` inspecciona el proveedor real de credenciales que usar? boto3.
- Si `runtime.describe()` muestra `credentials_configured=false` pero el smoke test corre, la sesi?n boto3 encontr? credenciales por otra v?a: perfil, SSO, role, metadata service o sandbox administrado.
- Este notebook no usa OpenAI ni mock; solo Bedrock.

## Coverage API de este notebook


In [ ]:
api_coverage = [
    {"api": "lab.runtime(provider='bedrock-runtime')", "description": "Declara Bedrock Runtime como backend canonico."},
    {"api": "lab.scheduler", "description": "Declara limites de ejecucion antes de llamar al provider."},
    {"api": "RuntimeConfig.describe", "description": "Expone provider, modelo, region y scheduler sin ejecutar inferencia."},
    {"api": "bedrock-runtime converse", "description": "Smoke opcional de lenguaje en AWS."},
    {"api": "bedrock-runtime invoke_model", "description": "Smoke opcional de embeddings en AWS."},
    {"api": "lab.human_result", "description": "Renderiza una vista humana sin llamar a AWS."},
    {"api": "shared scenario declared", "description": "Mantiene el mismo problema de fundamentals para comparacion 1:1."},
]

bedrock_dry_result = lab.RunResult(
    text="Bedrock Runtime quedo configurado para smoke tests opcionales.",
    data={"provider": "bedrock-runtime", "model": LANGUAGE_MODEL_ID, "region": REGION},
    final={"provider": "bedrock-runtime", "ready_for_smoke": RUN_BEDROCK_SMOKE_TESTS},
    engine="bedrock-runtime",
    model=LANGUAGE_MODEL_ID,
    mode="provider-smoke",
)

lab.human_result(bedrock_dry_result, pretty=False)
lab.show({"notebook": "00_runtime_bedrock_provider_api.ipynb", "api_coverage": api_coverage})


## S?mbolos API explicados

Este notebook se alinea con `docs/API.md` y ense?a estos s?mbolos p?blicos:

- `lab.runtime(provider="bedrock-runtime")`: Runtime expl?cito para Bedrock Runtime.
- `BedrockRuntimeClient`: Cliente p?blico de runtime Bedrock usado por la capa provider.
- `DEFAULT_EMBEDDING_MODEL_ID`: Modelo default p?blico para embeddings Bedrock.
- `boto3_session_snapshot`: Diagn?stico real de sesi?n boto3, m?s fiel que variables sueltas.
- `aws_environment_snapshot`: Snapshot seguro de se?ales AWS visibles.
- `repair_ada_credential_chain`: Utilidad p?blica para ambientes ADA con cadena AWS especial.
- `BEDROCK_RUNTIME_ENGINE`: Nombre can?nico del engine Bedrock.
- `RunResult`: Envelope usado para presentar smoke results con `human_result`.

